In [3]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import re
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION PARAMETERS
# ============================================================
RUN_SEL = None
MAX_DEBUG_PLOTS = 0
PLOT_ATTENUATION = True

# --- selezione produzione da digitalizzare (aggiornato per i tank lunghi) ---
DATA_DIR   = "data"                 # cartella ROOT e output .npz (era "build")
GEO_FILTER = "single_wbc2_long"     # digitalizza solo i ROOT con questa stringa nel nome ("" = tutti)
N_CHANNELS = 2                      # PMT per evento (single=2 -> ch_0/ch_1; triple=6)
SINGLE_CH_A, SINGLE_CH_B = 0, 1     # coppia PMT per l'analisi attenuazione (geometria single)

# ============================================================
# DIGITIZER, PMT & CABLE PARAMETERS
# ============================================================
SAMPLE_RATE_GS = 2.5
DT_NS = 1.0 / SAMPLE_RATE_GS
RECORD_LENGTH = 1024
ADC_BITS = 12
V_PP_MV = 1000.0
BASELINE_ADC = 3800
NOISE_RMS_ADC = 3.5

TRIGGER_OFFSET_NS = 150.0

G = 2.0e6
R = 50.0
e_charge = 1.602e-19
tr_pmt = 0.6
TTS_FWHM_NS = 0.3
TTS_SIGMA_NS = TTS_FWHM_NS / 2.355
SPE_RES = 0.40

CABLE_LENGTH_M = 20.0
ATTENUATION_DB = (15.0 / 100.0) * CABLE_LENGTH_M
TRANSMISSION_FACTOR = 10.0 ** (-ATTENUATION_DB / 20.0)

tr_cable = 0.14 * CABLE_LENGTH_M
tr_total = np.sqrt(tr_pmt**2 + tr_cable**2)
sigma_total = tr_total / 2.22

amp_SPE_mV_ideal = - (e_charge * G * R) / (sigma_total * 1e-9 * np.sqrt(2 * np.pi)) * 1e3
amp_SPE_mV_attenuated = amp_SPE_mV_ideal * TRANSMISSION_FACTOR

QUICK_INT_START_NS = 250.0
QUICK_INT_END_NS   = 410.0

DEBUG_XLIM_START_NS = 250.0
DEBUG_XLIM_END_NS   = 410.0

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def mv_to_adc(voltage_mv):
    lsb = V_PP_MV / (2**ADC_BITS)
    adc_float = BASELINE_ADC + (voltage_mv / lsb)
    adc_noisy = adc_float + np.random.normal(0, NOISE_RMS_ADC, len(adc_float))
    return np.clip(np.round(adc_noisy), 0, (2**ADC_BITS)-1).astype(np.uint16)

def get_quantum_efficiency(wavelengths_nm):
    wl_points = np.array([200.0, 250.0, 300.0, 350.0, 380.0, 400.0, 420.0,
                          450.0, 500.0, 550.0, 600.0, 650.0, 700.0])
    qe_points = np.array([0.18,  0.25,  0.30,  0.34,  0.35,  0.34,  0.32,
                          0.28,  0.18,  0.08,  0.03,  0.005, 0.00])
    return np.interp(wavelengths_nm, wl_points, qe_points, left=0.0, right=0.0)

def discover_tree(file_handle):
    # preferisci l'albero dei fotoni (photon-by-photon): quello con Arrival_Time_ns/PMT_ID
    trees = [k.split(";")[0] for k in file_handle.keys() if hasattr(file_handle[k], "keys")]
    for tn in trees:
        br = list(file_handle[tn].keys())
        if "Arrival_Time_ns" in br and "PMT_ID" in br:
            return tn
    return trees[0] if trees else None

def parse_scan_position_from_name(basename):
    # formato nuovo: ...X<val>mm_Y<val>mm... con 'm' = segno meno (es. Xm1500mm)
    def _conv(sv):
        sv = str(sv)
        return -float(sv[1:]) if sv.lower().startswith('m') else float(sv)

    m = re.search(r'X(m?\d+\.?\d*)mm_Y(m?\d+\.?\d*)mm', basename, re.IGNORECASE)
    if m:
        x_val = _conv(m.group(1)); y_val = _conv(m.group(2))
    else:
        # retro-compatibilita': vecchio ordine Y..._X...
        m2 = re.search(r'Y([+-]?\d+\.?\d*)_X([+-]?\d+\.?\d*)', basename, re.IGNORECASE)
        if not m2:
            return {"recognized": False, "y_val": np.nan, "x_val": np.nan,
                    "y_cm": np.nan, "x_cm": np.nan,
                    "mod_name": "Unknown / Background run", "ch_A": None, "ch_B": None}
        y_val = float(m2.group(1)); x_val = float(m2.group(2))

    y_cm = y_val / 10.0
    x_cm = x_val / 10.0

    if N_CHANNELS == 2:                       # geometria single: due PMT -> ch_0/ch_1
        mod_name = "Single tank (long)"
        ch_A, ch_B = SINGLE_CH_A, SINGLE_CH_B
    elif y_cm < -15.0:
        mod_name = "Bottom Module (Bundle 37, Mylar)"; ch_A, ch_B = 0, 1
    elif abs(y_cm) <= 15.0:
        mod_name = "Middle Module (Comb 37, HDPE)";    ch_A, ch_B = 2, 3
    else:
        mod_name = "Top Module (Bundle 18, Mylar)";    ch_A, ch_B = 4, 5

    return {
        "recognized": True,
        "y_val": y_val, "x_val": x_val, "y_cm": y_cm, "x_cm": x_cm,
        "mod_name": mod_name, "ch_A": ch_A, "ch_B": ch_B
    }

def read_root_data(file_path, basename):
    file = uproot.open(file_path)

    tree_name = discover_tree(file)
    if tree_name is None:
        raise RuntimeError(f"Nessun TTree trovato in {basename}. Keys: {file.keys()}")

    tree = file[tree_name]
    available_branches = list(tree.keys())

    print(f"    [INFO] Using tree: {tree_name}")
    print(f"    [INFO] Available branches: {available_branches}")

    old_format_branches = ["EventID", "Arrival_Time_ns", "PMT_ID", "E_Hit_eV"]

    # Branch names del nuovo formato: qui gestiamo varie possibilità comuni
    # perché i nomi effettivi dipendono da come è stata costruita la ntupla.
    candidate_new_sets = [
        ["Einit", "TrackLength", "Edep", "NCerenkovProd", "NCerenkovEntrati",
         "FirstHitTime", "PMT0", "PMT1", "PMT2", "PMT3", "PMT4", "PMT5", "EventType"],

        ["fEinit", "fTrackLength", "fEdep", "fNCerenkovProd", "fNCerenkovEntrati",
         "fFirstHitTime", "fPMTHits0", "fPMTHits1", "fPMTHits2", "fPMTHits3", "fPMTHits4", "fPMTHits5", "fEventType"],

        ["Einit", "TrackLength", "Edep", "NCerenkovProd", "NCerenkovEntrati",
         "FirstHitTime", "PMTHits_0", "PMTHits_1", "PMTHits_2", "PMTHits_3", "PMTHits_4", "PMTHits_5", "EventType"],
    ]

    if all(b in available_branches for b in old_format_branches):
        df = tree.arrays(old_format_branches, library="pd")
        return "old_photon", df, tree_name, available_branches

    for branch_set in candidate_new_sets:
        if all(b in available_branches for b in branch_set):
            df = tree.arrays(branch_set, library="pd")
            df = df.reset_index(drop=True)
            df["EventID"] = np.arange(len(df), dtype=int)
            return "new_event", df, tree_name, available_branches

    # Fallback: se esiste EventType, consideriamolo nuovo formato anche se
    # i nomi degli altri branch non sono ancora esattamente mappati
    if "EventType" in available_branches or "fEventType" in available_branches:
        eventtype_name = "EventType" if "EventType" in available_branches else "fEventType"
        df = tree.arrays([eventtype_name], library="pd")
        df = df.reset_index(drop=True)
        df["EventID"] = np.arange(len(df), dtype=int)
        return "new_event_partial", df, tree_name, available_branches

    raise RuntimeError(
        f"Formato ROOT non riconosciuto in {basename}. Branches trovati: {available_branches}"
    )

# ============================================================
# BATCH READING AND OUTPUT PREPARATION
# ============================================================
input_dir = DATA_DIR
root_files = glob.glob(os.path.join(input_dir, "*.root"))
if GEO_FILTER:
    root_files = [f for f in root_files if GEO_FILTER in os.path.basename(f)]

output_dir = DATA_DIR
os.makedirs(output_dir, exist_ok=True)

if len(root_files) == 0:
    print(f"No .root files found in {input_dir}. Please check the path.")
    raise SystemExit(1)

print(f"Found {len(root_files)} ROOT files. Starting massive digitization...")
print(f" -> Sample rate: {SAMPLE_RATE_GS} GS/s  (dt = {DT_NS:.3f} ns/sample, record {RECORD_LENGTH} samples = {RECORD_LENGTH*DT_NS:.1f} ns)")
print(f" -> Trigger offset: {TRIGGER_OFFSET_NS} ns  (signal centroid expected around sample {int(TRIGGER_OFFSET_NS/DT_NS)})")
print(f" -> PMT: Hamamatsu H14603-103 (G={G:.1e}, TTS={TTS_FWHM_NS} ns FWHM, SPE res={SPE_RES*100:.0f}%)")
print(f" -> Simulated Cable: {CABLE_LENGTH_M}m RG-58 (-{ATTENUATION_DB:.1f} dB, Tr_cable={tr_cable:.1f} ns)")

time_axis = np.arange(0, RECORD_LENGTH * DT_NS, DT_NS)
mock_catalog = []
events_summary = []
current_run_id = 1000
debug_plots_saved = 0
quick_analysis_data = []

window_start = int(QUICK_INT_START_NS / DT_NS)
window_end   = int(QUICK_INT_END_NS / DT_NS)

for file_path in sorted(root_files):

    if RUN_SEL is not None and current_run_id != RUN_SEL:
        current_run_id += 1
        continue

    basename = os.path.basename(file_path)
    info = parse_scan_position_from_name(basename)

    if info["recognized"]:
        print(f" -> Processing Run {current_run_id} | Y={info['y_val']} mm, X={info['x_val']} mm | Sector={info['mod_name']}")
    else:
        print(f" -> Processing Run {current_run_id} | File={basename} | Sector={info['mod_name']}")

    events_data_per_file = {}

    try:
        data_format, df, tree_name, available_branches = read_root_data(file_path, basename)
    except Exception as e:
        print(f"    [!] ROOT tree reading error: {e}")
        events_summary.append({
            'File_Root': basename,
            'Format': 'Read Error',
            'Valid_Events_Min_1_Photon': "Read Error"
        })
        current_run_id += 1
        continue

    # ========================================================
    # NUOVO FORMATO EVENT-BY-EVENT: per ora solo apertura + EventType
    # ========================================================
    if data_format in ["new_event", "new_event_partial"]:
        print("    [INFO] New event-by-event format detected.")
        print("    [INFO] Photon-by-photon digitization is currently skipped for this file.")

        if "EventType" in df.columns:
            et_col = "EventType"
        elif "fEventType" in df.columns:
            et_col = "fEventType"
        else:
            et_col = None

        if et_col is not None:
            print("    [INFO] EventType distribution:")
            print(df[et_col].value_counts(dropna=False).sort_index())
        else:
            print("    [WARNING] EventType branch not loaded, but new format was detected.")

        events_summary.append({
            'File_Root': basename,
            'Format': data_format,
            'Tree_Name': tree_name,
            'Num_Events': len(df),
            'Valid_Events_Min_1_Photon': "N/A (new format)",
            'Has_EventType': et_col is not None
        })

        mock_catalog.append({
            'run': current_run_id,
            'descrizione': f"New-format ROOT file: {basename}",
            'status': 'opened_only',
            'file_npz': 'not_generated',
            'module_hit': info['mod_name'],
            'valid_events': len(df)
        })

        current_run_id += 1
        continue

    # ========================================================
    # VECCHIO FORMATO PHOTON-BY-PHOTON: logica originale
    # ========================================================
    if df.empty:
        print("    [!] No photons recorded in this run.")
        events_summary.append({
            'File_Root': basename,
            'Format': data_format,
            'Valid_Events_Min_1_Photon': 0
        })
        current_run_id += 1
        continue

    event_ids = np.sort(df.EventID.unique())
    num_events_with_photons = len(event_ids)

    events_summary.append({
        'File_Root': basename,
        'Format': data_format,
        'Valid_Events_Min_1_Photon': num_events_with_photons
    })

    qa_list_run = []
    qb_list_run = []

    for ev_id in event_ids:
        event_dict = {}
        photon_counts_debug = {}
        hits_ev = df[df.EventID == ev_id]

        for pmt_id in range(N_CHANNELS):
            photons_pmt = hits_ev[hits_ev.PMT_ID == pmt_id]
            photon_counts_debug[pmt_id] = len(photons_pmt)

            if photons_pmt.empty:
                event_dict[f'ch_{pmt_id}'] = mv_to_adc(np.zeros_like(time_axis))
                continue

            arrival_times = photons_pmt.Arrival_Time_ns.values
            energies_ev   = photons_pmt.E_Hit_eV.values

            wavelengths_nm = 1240.0 / energies_ev
            qe_probs = get_quantum_efficiency(wavelengths_nm)
            accepted_mask = np.random.rand(len(qe_probs)) < qe_probs
            accepted_arrival_times = arrival_times[accepted_mask]

            voltage = np.zeros_like(time_axis)
            for t_hit in accepted_arrival_times:
                tts_jitter = np.random.normal(0.0, TTS_SIGMA_NS)
                t_shifted = t_hit + TRIGGER_OFFSET_NS + tts_jitter

                spe_gain_factor = np.random.normal(1.0, SPE_RES)
                if spe_gain_factor <= 0:
                    continue

                voltage += amp_SPE_mV_attenuated * spe_gain_factor * np.exp(
                    -0.5 * ((time_axis - t_shifted) / sigma_total)**2
                )

            event_dict[f'ch_{pmt_id}'] = mv_to_adc(voltage)

        if PLOT_ATTENUATION and info["recognized"]:
            lsb = V_PP_MV / (2**ADC_BITS)
            sigA_mV = (event_dict[f'ch_{info["ch_A"]}'].astype(float) - BASELINE_ADC) * lsb
            sigB_mV = (event_dict[f'ch_{info["ch_B"]}'].astype(float) - BASELINE_ADC) * lsb

            qA_val = -np.sum(sigA_mV[window_start:window_end])
            qB_val = -np.sum(sigB_mV[window_start:window_end])

            if qA_val > 0 and qB_val > 0:
                qa_list_run.append(qA_val)
                qb_list_run.append(qB_val)

        if debug_plots_saved < MAX_DEBUG_PLOTS:
            fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
            fig.suptitle(f"DEBUG OSCILLOSCOPE: Run {current_run_id} - Event {ev_id} ({basename})", fontsize=16, fontweight='bold')
            lsb = V_PP_MV / (2**ADC_BITS)

            for pmt_id in range(N_CHANNELS):
                ax = axes[pmt_id // 3, pmt_id % 3]
                signal_adc = event_dict[f'ch_{pmt_id}']
                num_ph = photon_counts_debug[pmt_id]
                signal_mv = (signal_adc.astype(float) - BASELINE_ADC) * lsb

                ax.plot(time_axis, signal_mv, color='darkcyan', linewidth=1.5)
                ax.axhline(0.0, color='red', linestyle='--', alpha=0.7, label="Baseline (0 mV)")
                ax.axvline(TRIGGER_OFFSET_NS, color='magenta', linestyle=':', alpha=0.7,
                           label=f"Trigger ref ({TRIGGER_OFFSET_NS:.0f} ns)")
                ax.set_title(f"PMT {pmt_id} ({num_ph} Photons)", fontsize=12)
                ax.set_xlim(DEBUG_XLIM_START_NS, DEBUG_XLIM_END_NS)
                ax.grid(True, linestyle=':', alpha=0.6)

                if pmt_id >= 3:
                    ax.set_xlabel("Time (ns)")
                if pmt_id % 3 == 0:
                    ax.set_ylabel("Amplitude (mV)")
                if pmt_id == 0:
                    ax.legend(loc='upper right', fontsize='small')

            plt.tight_layout()
            plt.show()
            debug_plots_saved += 1

        events_data_per_file[f'run_{current_run_id}_ev_{ev_id}'] = event_dict

    if PLOT_ATTENUATION and info["recognized"] and len(qa_list_run) > 0:
        qa_arr = np.array(qa_list_run)
        qb_arr = np.array(qb_list_run)

        log_ratios_raw = np.log(qa_arr / qb_arr)
        mean_log_ratio = np.mean(log_ratios_raw)
        sem_log_ratio = np.std(log_ratios_raw) / np.sqrt(len(log_ratios_raw))

        mean_qa, sem_qa = np.mean(qa_arr), np.std(qa_arr) / np.sqrt(len(qa_arr))
        mean_qb, sem_qb = np.mean(qb_arr), np.std(qb_arr) / np.sqrt(len(qb_arr))

        quick_analysis_data.append({
            'Module': info["mod_name"],
            'Y_cm': info["y_cm"],
            'X_cm': info["x_cm"],
            'Log_Ratio': mean_log_ratio,
            'Err_Log_Ratio': sem_log_ratio,
            'Q_A': mean_qa,
            'Err_Q_A': sem_qa,
            'Q_B': mean_qb,
            'Err_Q_B': sem_qb,
            'Num_Events': len(qa_list_run)
        })

    npz_filename = f"{os.path.splitext(basename)[0]}.npz"
    npz_path = os.path.join(output_dir, npz_filename)
    np.savez_compressed(npz_path, **events_data_per_file)

    mock_catalog.append({
        'run': current_run_id,
        'descrizione': f"Tank Scan Y={info['y_cm']} cm X={info['x_cm']} cm" if info["recognized"] else basename,
        'status': 'ok',
        'file_npz': npz_filename,
        'module_hit': info['mod_name'],
        'valid_events': num_events_with_photons
    })

    current_run_id += 1

# ============================================================
# QUICK ATTENUATION ANALYSIS
# ============================================================
if PLOT_ATTENUATION and len(quick_analysis_data) > 0:
    print("\n[OK] Generazione Plot Log-Ratio e Cariche per la nuova vasca monolitica...")
    df_qa = pd.DataFrame(quick_analysis_data)

    mods_found = sorted(df_qa['Module'].unique())

    style_map = {
        'Top Module (Bundle 18, Mylar)': {'color': 'blue',  'marker': 'o'},
        'Middle Module (Comb 37, HDPE)': {'color': 'green', 'marker': 's'},
        'Bottom Module (Bundle 37, Mylar)': {'color': 'red', 'marker': '*'}
    }

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("Quick Analysis: Charge & Attenuation vs Beam X Position (Separated by Hit Sector)", fontsize=16, fontweight='bold')

    for mod in mods_found:
        df_mod = df_qa[df_qa['Module'] == mod].sort_values(by='X_cm')
        if len(df_mod) < 2:
            continue

        X_arr = df_mod['X_cm'].values
        y_val_mod = df_mod['Y_cm'].iloc[0]

        c_dict = style_map.get(mod, {'color': 'gray', 'marker': 'x'})
        c_col, c_mark = c_dict['color'], c_dict['marker']

        Y_log = df_mod['Log_Ratio'].values
        Err_log = df_mod['Err_Log_Ratio'].values
        axes[0].errorbar(X_arr, Y_log, yerr=Err_log, fmt=c_mark, color=c_col, markersize=8,
                         label=f'{mod} (Y={y_val_mod}cm)')

        slope, intercept = np.polyfit(X_arr, Y_log, 1)
        lambda_eff = -2.0 / slope
        axes[0].plot(X_arr, slope*X_arr + intercept, linestyle='--', color=c_col, alpha=0.6,
                     label=fr'Fit [$\lambda$ = {lambda_eff:.0f} cm]')

        Y_qa = df_mod['Q_A'].values
        Err_qa = df_mod['Err_Q_A'].values
        axes[1].errorbar(X_arr, Y_qa, yerr=Err_qa, fmt=c_mark, color=c_col, markersize=8, label=f'{mod}')

        Y_qb = df_mod['Q_B'].values
        Err_qb = df_mod['Err_Q_B'].values
        axes[2].errorbar(X_arr, Y_qb, yerr=Err_qb, fmt=c_mark, color=c_col, markersize=8, label=f'{mod}')

    axes[0].set_title(r"Charge Log-Ratio $\ln(Q_A / Q_B)$", fontsize=14)
    axes[0].set_ylabel(r"Media $\ln(Q_A / Q_B)$")
    axes[1].set_title(r"PMT Left Signal Area ($Q_A$)", fontsize=14)
    axes[1].set_ylabel(r"Media $Q_A$")
    axes[2].set_title(r"PMT Right Signal Area ($Q_B$)", fontsize=14)
    axes[2].set_ylabel(r"Media $Q_B$")

    for ax in axes:
        ax.set_xlabel("Beam X Position (cm)")
        ax.grid(True, linestyle=':', alpha=0.7)
        ax.legend(fontsize=9)

    fig.tight_layout()
    plt.show()

# ============================================================
# SAVE CSV OUTPUTS
# ============================================================
if RUN_SEL is None:
    csv_path = os.path.join(output_dir, "mc_runs_catalog.csv")
    pd.DataFrame(mock_catalog).to_csv(csv_path, index=False)
    print(f"\n[OK] Run catalog saved to: {csv_path}")

    summary_path = os.path.join(output_dir, "eventi_con_fotoni_summary.csv")
    pd.DataFrame(events_summary).to_csv(summary_path, index=False)
    print(f"[OK] Valid events summary saved to: {summary_path}")


Found 1 ROOT files. Starting massive digitization...
 -> Sample rate: 2.5 GS/s  (dt = 0.400 ns/sample, record 1024 samples = 409.6 ns)
 -> Trigger offset: 150.0 ns  (signal centroid expected around sample 375)
 -> PMT: Hamamatsu H14603-103 (G=2.0e+06, TTS=0.3 ns FWHM, SPE res=40%)
 -> Simulated Cable: 20.0m RG-58 (-3.0 dB, Tr_cable=2.8 ns)
 -> Processing Run 1000 | File=test_debug.root | Sector=Unknown / Background run
    [INFO] Using tree: Eventi
    [INFO] Available branches: ['E_init_MeV', 'TrackL_water_mm', 'Edep_water_MeV', 'N_Cerenkov_prod', 'N_Cerenkov_entrati', 'T_FirstHit_ns', 'Hits_L_Bot', 'Hits_R_Bot', 'Hits_L_Mid', 'Hits_R_Mid', 'Hits_L_Top', 'Hits_R_Top', 'EventType']
    [INFO] New event-by-event format detected.
    [INFO] Photon-by-photon digitization is currently skipped for this file.
    [INFO] EventType distribution:
EventType
1    100000
Name: count, dtype: int64

[OK] Run catalog saved to: build/mc_runs_catalog.csv
[OK] Valid events summary saved to: build/eventi